# FloodLite D2 — combined-dataset training (reduced grid)

Trains **teacher + 3 students × {none, comb} × 1 fold** on *flood-combined-fsegnet*
(set `FOLD` below; run the notebook three times for folds 0–2 ≈ 21 runs total).

**Prerequisites**
1. *Add Data* → search `flood-combined-fsegnet` (tasfianoorchowdhury) → attach.
2. The FSSD↔combined dedup gate (`scripts/check_fssd_overlap.py`) must have passed
   — see `runs/fssd_overlap_report.json` in the repo.
3. GPU on (T4). One fold ≈ 7 training runs; if the 12 h session limit hits,
   re-run with the previous version's `/kaggle/working/runs` attached and copied
   back in — `--skip_if_exists` resumes from existing checkpoints.

Split semantics (from `manifest.csv`): train = kaggle(fold≠F) + FloodNet train+val;
val = kaggle fold F (duplicate-aware); test = FloodNet test (424, fixed);
external = AIFloodSense (94, zero-shot). Training uses the source-balanced sampler.
Quantization/latency legs are skipped here (architecture properties already
characterised on FSSD; D3 quantizes the combined-trained Lite0 separately).

In [ ]:
FOLD = 0          # <-- set to 0, 1, or 2 (one fold per session)
EPOCHS = 35
BATCH = 8

In [ ]:
!git clone --depth 1 https://github.com/NoorTasfia/flood-segmentation.git /kaggle/working/repo
%cd /kaggle/working/repo/code
%pip install -q "segmentation_models_pytorch>=0.4.0,<0.6" "timm>=0.9.7,<2.0" albumentations==1.4.18 torchprofile

In [ ]:
from pathlib import Path
# Locate the combined dataset root (the folder that holds manifest.csv)
candidates = list(Path('/kaggle/input').glob('**/manifest.csv'))
assert candidates, 'Attach the flood-combined-fsegnet dataset first (Add Data).'
DATA_ROOT = candidates[0].parent
print('DATA_ROOT =', DATA_ROOT)

# Resume support: if a previous session's runs/ was attached as input, copy it in
import shutil
prev = list(Path('/kaggle/input').glob('**/runs_combined/*.pt'))
if prev:
    dest = Path('/kaggle/working/runs_combined')
    dest.mkdir(parents=True, exist_ok=True)
    for p in prev:
        shutil.copy(p, dest / p.name)
    print(f'Resumed {len(prev)} checkpoints from a previous session')

In [ ]:
!python scripts/run_full_experiment.py \
    --dataset combined \
    --data_root {DATA_ROOT} \
    --fold {FOLD} --folds 1 \
    --kd_configs none comb \
    --skip_quant --skip_if_exists \
    --epochs_teacher {EPOCHS} --epochs_student {EPOCHS} \
    --batch_size {BATCH} \
    --out_dir /kaggle/working/runs_combined

In [ ]:
# Keep results + checkpoints as session output (checkpoints enable resume + D3)
import json
from pathlib import Path
res = Path('/kaggle/working/runs_combined/results.json')
if res.exists():
    r = json.loads(res.read_text())
    for fold, blocks in r.items():
        print(f"== {fold} ==")
        t = blocks['teacher']
        for split in ('val', 'test', 'external'):
            print(f"  teacher {split}: IoU {t[split]['iou_mean']:.4f} +/- {t[split]['iou_std']:.4f}")
        for name, m in blocks['students'].items():
            print(f"  {name}: val {m['val']['iou_mean']:.4f}  test {m['test']['iou_mean']:.4f}  ext {m['external']['iou_mean']:.4f}")